# Robustness checks for the final pooled model

This notebook challenges the fixed pooled market-plus-player model without selecting new features. It uses only the five development seasons and leaves 2025/26 untouched.

It displays the existing raw-market, recalibrated-market and final-model comparison, then runs three new checks: major feature-block removals, one fixed within-league/season player shuffle, and one deliberately invalid hindsight model. The hindsight model is a pipeline warning only and must never appear in the legitimate model ranking.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'scotland_research').is_dir()
)
RESEARCH_DIR = PROJECT_ROOT / 'scotland_research'
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

from constants import DEFAULT_EVALUATION_DIR, DEFAULT_MODEL_DATASET
from data.load_model_dataset import load_dataset
from evaluation.multi_league import (
    POOLED_SCOPE,
    league_effect_column_names,
    run_multi_league_walk_forward,
)
from evaluation.report import write_csv_atomic
from robustness_checks import (
    HINDSIGHT_FEATURE,
    SHUFFLE_SEED,
    add_hindsight_final_points,
    add_within_league_season_shuffle,
    build_primary_summary,
    build_robustness_factories,
    settings_record,
    validate_same_matches,
)
from selected_features import load_selected_features

OUTPUT_DIR = RESEARCH_DIR / 'visuals' / 'robustness_checks'
TABLES_DIR = OUTPUT_DIR / 'tables'
FIGURES_DIR = OUTPUT_DIR / 'figures'
MAIN_RESULTS_DIR = DEFAULT_EVALUATION_DIR

print(f'Project root: {PROJECT_ROOT}')
print(f'Outputs: {OUTPUT_DIR}')

## 1. Load the fixed specification

The selected feature file is loaded with its registered checksum. The notebook stops if the feature blocks no longer match the fixed market-plus-player model.

In [ ]:
selected = load_selected_features()
final_features = selected.by_model['market_plus_player_form']
run_settings = dict(settings_record(final_features))
run_settings['selected_features_sha256'] = selected.semantic_sha256
run_settings['main_results_directory'] = str(MAIN_RESULTS_DIR)
run_settings['model_dataset'] = str(DEFAULT_MODEL_DATASET)

pd.DataFrame(
    [(block, ', '.join(features)) for block, features in run_settings['feature_blocks'].items()],
    columns=['feature_block', 'fixed_features'],
)

## 2. Display the existing primary comparison

These models are not retrained here. The notebook reads the results produced by `evaluate_models.py`.

In [ ]:
required_main_files = {
    'equal_league_metrics': MAIN_RESULTS_DIR / 'equal_league_metrics.csv',
    'fold_equal_league_metrics': MAIN_RESULTS_DIR / 'fold_equal_league_metrics.csv',
    'overall_league_metrics': MAIN_RESULTS_DIR / 'overall_league_metrics.csv',
    'predictions': MAIN_RESULTS_DIR / 'predictions.csv',
}
missing_main_files = [str(path) for path in required_main_files.values() if not path.exists()]
if missing_main_files:
    raise FileNotFoundError(
        'Run scotland_research/evaluate_models.py first. Missing: '
        + ', '.join(missing_main_files)
    )

main_equal = pd.read_csv(required_main_files['equal_league_metrics'])
main_fold = pd.read_csv(required_main_files['fold_equal_league_metrics'])
main_league = pd.read_csv(required_main_files['overall_league_metrics'])
main_predictions = pd.read_csv(required_main_files['predictions'])
primary_names = ['closing_market', 'recalibrated_market', 'market_plus_player_form']
existing_primary = main_equal[
    main_equal['training_scope'].eq('pooled')
    & main_equal['model'].isin(primary_names)
][['model', 'matches', 'log_loss', 'brier_score', 'accuracy']].copy()
existing_primary['model'] = pd.Categorical(
    existing_primary['model'], categories=primary_names, ordered=True
)
existing_primary.sort_values('model')

## 3. Prepare the two deliberately altered datasets

The shuffle moves the four genuine player-performance columns together within each league and season. Market inputs, team context, results and match identities stay fixed.

The hindsight feature uses each team's final points per match for the complete season, including matches that had not happened at prediction time. It is intentionally invalid and exists only to demonstrate the effect of future information. The source dataset and selected-features file are never modified.

In [ ]:
dataset = load_dataset(DEFAULT_MODEL_DATASET)
if HINDSIGHT_FEATURE in dataset.columns:
    raise ValueError('The legitimate model dataset already contains the forbidden hindsight column')

check_dataset = add_within_league_season_shuffle(dataset, seed=SHUFFLE_SEED)
check_dataset = add_hindsight_final_points(check_dataset)
factories = build_robustness_factories(
    final_features,
    league_effect_column_names(),
)

print(f'Development rows: {len(check_dataset):,}')
print('New checks:', ', '.join(name for name in factories if name != 'closing_market'))

## 4. Run the new checks

Only pooled models are fitted. Every check uses the same chronological development folds and equal-league training weights as the main evaluation.

In [ ]:
check_result = run_multi_league_walk_forward(
    check_dataset,
    pooled_factories=factories,
    league_specific_factories={},
    scopes=(POOLED_SCOPE,),
)
validate_same_matches(main_predictions, check_result.predictions)
print('Match audit passed: the main and robustness comparisons use identical matches.')

## 5. Save separate, auditable outputs

In [ ]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

summary = build_primary_summary(main_equal, check_result.equal_league_metrics)
combined_fold = pd.concat([
    main_fold[
        main_fold['training_scope'].eq('pooled')
        & main_fold['model'].isin(primary_names)
    ],
    check_result.fold_equal_league_metrics[
        ~check_result.fold_equal_league_metrics['model'].eq('closing_market')
    ],
], ignore_index=True)
combined_league = pd.concat([
    main_league[
        main_league['training_scope'].eq('pooled')
        & main_league['model'].isin(primary_names)
    ],
    check_result.overall_league_metrics[
        ~check_result.overall_league_metrics['model'].eq('closing_market')
    ],
], ignore_index=True)

table_outputs = {
    'primary_comparison': existing_primary,
    'robustness_summary': summary,
    'robustness_by_fold': combined_fold,
    'robustness_by_league': combined_league,
    'robustness_predictions': check_result.predictions,
    'training_weight_audit': check_result.training_weight_audit,
}
for name, frame in table_outputs.items():
    write_csv_atomic(frame, TABLES_DIR / f'{name}.csv')

settings_path = TABLES_DIR / 'run_settings.json'
settings_path.write_text(json.dumps(run_settings, indent=2), encoding='utf-8')
print(f'Saved {len(table_outputs)} tables and settings to {TABLES_DIR}')

## 6. Summary figure

Positive changes mean the check performed worse than the final model. For a removed block or shuffled players, that supports the final model. The hindsight bar is deliberately invalid and should be read only as a leakage warning.

In [ ]:
plot_data = summary[~summary['model'].eq('market_plus_player_form')].copy()
plot_data['change_per_1000'] = 1000 * plot_data['log_loss_change_vs_final']
label_map = {
    'closing_market': 'Raw market',
    'recalibrated_market': 'Recalibrated market',
    'without_attacking_output': 'Without attacking output',
    'without_defensive_output': 'Without defensive output',
    'without_player_ratings': 'Without player ratings',
    'without_team_strength_context': 'Without team-strength context',
    'shuffled_player_features': 'Shuffled player features',
    'deliberate_hindsight_model': 'INVALID: complete-season hindsight',
}
plot_data['label'] = plot_data['model'].map(label_map)
colors = [
    '#b2182b' if model == 'deliberate_hindsight_model'
    else '#666666' if model in {'closing_market', 'recalibrated_market'}
    else '#2166ac'
    for model in plot_data['model']
]
fig, axis = plt.subplots(figsize=(10, 6))
axis.barh(plot_data['label'], plot_data['change_per_1000'], color=colors)
axis.axvline(0, color='black', linewidth=1)
axis.set_xlabel('Log-loss change versus final pooled model (×1,000)')
axis.set_title('Compact robustness checks')
axis.spines[['top', 'right']].set_visible(False)
axis.invert_yaxis()
fig.tight_layout()
figure_path = FIGURES_DIR / 'robustness_log_loss_changes.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {figure_path}')

In [ ]:
summary[[
    'model',
    'matches',
    'log_loss',
    'log_loss_change_vs_final',
    'brier_score',
    'brier_change_vs_final',
    'interpretation',
]]

## Reading the result

- A positive removal change means that feature block helped the final model.
- A positive shuffle change means correctly matched player information beat the placebo.
- A negative hindsight change shows how future information can create an artificial improvement. It is not evidence for a usable model.
- Inspect `robustness_by_fold.csv` and `robustness_by_league.csv`; a tiny overall gain that reverses repeatedly is not robust.
- These are development checks only. Do not change the fixed individual features merely because one check looks favourable or unfavourable.